# Math Practice

> Added later to fill an originally-empty notebook (see repo root README). Written to match the spirit of the other `homework_0` warm-up exercises — a refresher on the linear algebra and calculus that the rest of the coursework leans on. Not executed in this environment (no notebook kernel run here), but every cell is plain NumPy/SymPy and runs as-is.


## 1. Vectors & matrices

In [ ]:
import numpy as np

v = np.array([2, -1, 3])
w = np.array([1, 4, 0])

print("v + w      =", v + w)
print("v . w      =", np.dot(v, w))
print("||v||_2    =", np.linalg.norm(v))
print("v (x) w    =\n", np.outer(v, w))


In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[0, 1], [1, 0]])

print("A @ B =\n", A @ B)
print("A.T   =\n", A.T)
print("det(A) =", np.linalg.det(A))
print("inv(A) =\n", np.linalg.inv(A))


## 2. Derivatives & the chain rule

Every backprop step is really the chain rule applied to a computation graph. `sympy` lets us check hand-derived gradients symbolically before trusting a NumPy implementation.

In [ ]:
import sympy as sp

x = sp.symbols("x")
f = sp.sin(x**2) + sp.exp(2 * x)

df_dx = sp.diff(f, x)
print("f(x)  =", f)
print("f'(x) =", df_dx)


## 3. Gradient of a scalar loss w.r.t. a vector

In [ ]:
x1, x2, w1, w2, b = sp.symbols("x1 x2 w1 w2 b")

z = w1 * x1 + w2 * x2 + b          # linear layer, one neuron
y_hat = 1 / (1 + sp.exp(-z))        # sigmoid activation
y = sp.symbols("y")
loss = -(y * sp.log(y_hat) + (1 - y) * sp.log(1 - y_hat))  # binary cross-entropy

grad_w1 = sp.simplify(sp.diff(loss, w1))
print("dL/dw1 =", grad_w1)


Sanity-check the well-known simplified result `dL/dz = y_hat - y` for BCE + sigmoid:

In [ ]:
grad_z = sp.simplify(sp.diff(loss, z).subs(z, w1 * x1 + w2 * x2 + b))
print("dL/dz simplifies to:", grad_z)


## 4. Gradient descent on a toy loss surface

In [ ]:
def f(x, y):
    return (x - 3) ** 2 + (y + 1) ** 2

def grad_f(x, y):
    return np.array([2 * (x - 3), 2 * (y + 1)])

point = np.array([0.0, 0.0])
lr = 0.1

history = [point.copy()]
for step in range(30):
    point = point - lr * grad_f(*point)
    history.append(point.copy())

print("Converged to:", point, "  (expected minimum: [3, -1])")


In [ ]:
import matplotlib.pyplot as plt

history = np.array(history)
plt.plot(history[:, 0], history[:, 1], "o-")
plt.scatter([3], [-1], c="red", marker="x", s=100, label="minimum")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Gradient descent trajectory")
plt.legend()
plt.show()


## 5. Softmax and cross-entropy derivative

In [ ]:
def softmax(z):
    ez = np.exp(z - np.max(z))
    return ez / ez.sum()

logits = np.array([2.0, 1.0, 0.1])
probs = softmax(logits)
print("probs =", probs, " sum =", probs.sum())

# For softmax + categorical cross-entropy, dL/dz = probs - one_hot(label)
label = 0
one_hot = np.eye(len(logits))[label]
grad = probs - one_hot
print("dL/dz =", grad)
